# 🛒 Retail Price Optimization
### Load · Explore · Model · Optimize — using your own dataset
**Libraries:** Pandas · NumPy · Matplotlib · Seaborn · Scikit-learn · LightGBM

---
## 📋 Table of Contents
1. [Install & Import Libraries](#step1)
2. [Load & Validate Dataset](#step2)
3. [Exploratory Data Analysis (EDA)](#step3)
4. [Feature Engineering](#step4)
5. [Price Elasticity Analysis](#step5)
6. [Build Demand Prediction Models](#step6)
7. [Feature Importance](#step7)
8. [Price Optimization Algorithm](#step8)
9. [Visualize Optimization Curves](#step9)
10. [Business Impact Analysis](#step10)
11. [Key Insights & Summary](#step11)


---
## Step 1 — Install & Import Libraries <a id='step1'></a>
Run the pip install cell once if LightGBM is not already installed.


In [ ]:
# !pip install lightgbm

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import lightgbm as lgb
import warnings

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
np.random.seed(42)

print("✅ All libraries imported successfully!")


---
## Step 2 — Load & Validate Dataset <a id='step2'></a>

We load `retail_price_optimization_dataset.csv` directly.  
Put the CSV in the **same folder** as this notebook before running.

**Expected columns:**

| Column | Type | Description |
|---|---|---|
| `product_id` | int | Unique product identifier |
| `week` | int | Week number |
| `base_price` | float | Reference price per product |
| `price` | float | Actual selling price |
| `competitor_price` | float | Competitor's price |
| `marketing_spend` | float | Weekly marketing spend ($) |
| `inventory_level` | int | Units in stock |
| `promotion` | str | None / Discount / Bundle |
| `season` | str | Winter / Spring / Summer / Fall |
| `day_of_week` | str | Monday … Sunday |
| `quantity_sold` | int | Units sold (target variable) |
| `revenue` | float | price × quantity_sold |
| `profit` | float | (price − cost) × quantity_sold |
| `price_elasticity` | float | Pre-calculated elasticity |


In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
CSV_PATH = "retail_price_optimization_dataset.csv"   # change path if needed
df = pd.read_csv(CSV_PATH)

print(f"Shape            : {df.shape}")
print(f"Products         : {df['product_id'].nunique()}  "
      f"(IDs {int(df['product_id'].min())}-{int(df['product_id'].max())})")
print(f"Weeks covered    : {df['week'].min()} - {df['week'].max()}")
print(f"Missing values   :\n{df.isnull().sum()[df.isnull().sum() > 0]}")
df.head()


In [ ]:
# ── Clean ─────────────────────────────────────────────────────────────────────
# Fill missing promotion with 'None' (string)
df['promotion'] = df['promotion'].fillna('None')

# Drop rows missing critical numeric columns
required_cols = ['price', 'competitor_price', 'marketing_spend',
                 'inventory_level', 'quantity_sold']
before = len(df)
df.dropna(subset=required_cols, inplace=True)
print(f"Rows dropped (missing critical cols): {before - len(df)}")
print(f"Clean dataset shape: {df.shape}")


In [ ]:
# ── Statistical summary ────────────────────────────────────────────────────────
df.describe().round(2)


---
## Step 3 — Exploratory Data Analysis <a id='step3'></a>

Understand distributions and relationships before modelling.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].hist(df['price'], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].set(title='Price Distribution', xlabel='Price ($)', ylabel='Frequency')

axes[0, 1].hist(df['quantity_sold'], bins=50, color='lightcoral', edgecolor='black', alpha=0.7)
axes[0, 1].set(title='Quantity Sold Distribution', xlabel='Quantity', ylabel='Frequency')

axes[0, 2].hist(df['revenue'], bins=50, color='lightgreen', edgecolor='black', alpha=0.7)
axes[0, 2].set(title='Revenue Distribution', xlabel='Revenue ($)', ylabel='Frequency')

promo_counts = df['promotion'].value_counts()
axes[1, 0].bar(promo_counts.index, promo_counts.values,
               color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[1, 0].set(title='Promotion Type Distribution', ylabel='Count')

season_counts = df['season'].value_counts()
axes[1, 1].bar(season_counts.index, season_counts.values,
               color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[1, 1].set(title='Seasonal Distribution', ylabel='Count')

axes[1, 2].scatter(df['price'], df['quantity_sold'], alpha=0.3, color='purple', s=8)
z = np.polyfit(df['price'], df['quantity_sold'], 1)
axes[1, 2].plot(np.sort(df['price']), np.poly1d(z)(np.sort(df['price'])),
                'r--', lw=2, label='Trend')
axes[1, 2].set(title='Price vs Quantity Sold', xlabel='Price ($)', ylabel='Quantity Sold')
axes[1, 2].legend()

plt.suptitle('Retail Dataset — Exploratory Analysis', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
print("=== Avg Revenue & Profit by Season ===")
print(df.groupby('season')[['revenue','profit']].mean().round(2))
print("\n=== Avg Revenue & Profit by Promotion ===")
print(df.groupby('promotion')[['revenue','profit']].mean().round(2))
print("\n=== Products per Week (check balance) ===")
print(df.groupby('week')['product_id'].count().describe().round(1))


---
## Step 4 — Feature Engineering <a id='step4'></a>

We derive new features to make price relationships explicit for the model.

| New Feature | Formula | Meaning |
|---|---|---|
| `price_ratio` | price ÷ competitor_price | Are we cheaper or pricier? |
| `price_gap` | competitor_price − price | Absolute price difference |
| `price_premium` | (price − base_price) ÷ base_price | Markup over base |
| `marketing_per_unit` | marketing_spend ÷ quantity_sold | Cost per unit sold |
| `inventory_to_sales` | inventory_level ÷ quantity_sold | Stock-to-demand ratio |
| `low_inventory_flag` | inventory < 100 → 1 | Scarcity indicator |
| `elasticity_category` | binned elasticity | Sensitivity label |


In [ ]:
df_features = df.copy()

df_features['price_ratio']        = df_features['price'] / df_features['competitor_price']
df_features['price_gap']          = df_features['competitor_price'] - df_features['price']
df_features['price_premium']      = ((df_features['price'] - df_features['base_price'])
                                      / df_features['base_price'])
df_features['marketing_per_unit'] = (df_features['marketing_spend']
                                     / np.maximum(df_features['quantity_sold'], 1))
df_features['inventory_to_sales'] = (df_features['inventory_level']
                                     / np.maximum(df_features['quantity_sold'], 1))
df_features['low_inventory_flag'] = (df_features['inventory_level'] < 100).astype(int)

df_features['elasticity_category'] = pd.cut(
    df_features['price_elasticity'],
    bins=[-np.inf, -2, -1.5, -1, np.inf],
    labels=['High', 'Medium-High', 'Medium-Low', 'Low']
)

le_season = LabelEncoder()
le_promo  = LabelEncoder()
le_day    = LabelEncoder()

df_features['season_encoded']      = le_season.fit_transform(df_features['season'])
df_features['promotion_encoded']   = le_promo.fit_transform(df_features['promotion'])
df_features['day_of_week_encoded'] = le_day.fit_transform(df_features['day_of_week'])

print("✅ Feature engineering complete!")
print(f"Total columns: {df_features.shape[1]}")
df_features[['price_ratio','price_gap','price_premium',
             'marketing_per_unit','inventory_to_sales']].describe().round(3)


---
## Step 5 — Price Elasticity Analysis <a id='step5'></a>

**Price Elasticity** = % Δ Quantity ÷ % Δ Price

- **< −1** → Elastic: customers are sensitive; price rises hurt demand significantly  
- **> −1** → Inelastic: customers are less sensitive; good candidates for price increases


In [ ]:
elasticity_by_product = (
    df_features
    .groupby('product_id')
    .agg(
        Avg_Elasticity  = ('price_elasticity', 'mean'),
        Avg_Price       = ('price', 'mean'),
        Avg_Quantity    = ('quantity_sold', 'mean'),
        Avg_Revenue     = ('revenue', 'mean'),
        Avg_Profit      = ('profit', 'mean')
    )
    .round(2)
    .sort_values('Avg_Elasticity')
)

print("=== Top 10 Most Price-Sensitive Products ===")
print(elasticity_by_product.head(10))
print("\n=== Top 10 Least Price-Sensitive Products ===")
print(elasticity_by_product.tail(10))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

mean_e = df_features['price_elasticity'].mean()
axes[0, 0].hist(df_features['price_elasticity'], bins=50,
                color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(mean_e, color='red', ls='--', lw=2, label=f'Mean: {mean_e:.2f}')
axes[0, 0].set(title='Price Elasticity Distribution',
               xlabel='Price Elasticity', ylabel='Frequency')
axes[0, 0].legend()

axes[0, 1].scatter(elasticity_by_product['Avg_Price'],
                   elasticity_by_product['Avg_Elasticity'],
                   s=80, alpha=0.6, color='green')
axes[0, 1].set(title='Avg Price vs Price Elasticity',
               xlabel='Avg Price ($)', ylabel='Price Elasticity')
for idx in elasticity_by_product.head(5).index:
    axes[0, 1].annotate(f"P{idx}",
        (elasticity_by_product.loc[idx,'Avg_Price'],
         elasticity_by_product.loc[idx,'Avg_Elasticity']),
        fontsize=8)

rev_e    = df_features.groupby('elasticity_category')['revenue'].mean().sort_values()
colors_e = ['#1f77b4','#ff7f0e','#2ca02c','#d62728']
axes[1, 0].barh(rev_e.index, rev_e.values, color=colors_e)
axes[1, 0].set(title='Avg Revenue by Elasticity Category', xlabel='Avg Revenue ($)')
for i, v in enumerate(rev_e.values):
    axes[1, 0].text(v, i, f' ${v:,.0f}', va='center', fontweight='bold')

df_features.boxplot(column='price_elasticity', by='promotion', ax=axes[1, 1])
axes[1, 1].set(title='Price Elasticity by Promotion Type',
               xlabel='Promotion', ylabel='Elasticity')
plt.sca(axes[1, 1]); plt.xticks(rotation=45)
plt.suptitle('')

plt.suptitle('Price Elasticity Analysis', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


---
## Step 6 — Build Demand Prediction Models <a id='step6'></a>

Three models trained and compared:

| Model | Notes |
|---|---|
| Linear Regression | Baseline |
| Random Forest | Non-linear, robust |
| **LightGBM** ✅ | Best accuracy — gradient boosting |

**Target:** `quantity_sold`


In [ ]:
feature_cols = [
    'price', 'competitor_price', 'marketing_spend', 'inventory_level',
    'price_ratio', 'price_gap', 'price_premium', 'marketing_per_unit',
    'season_encoded', 'promotion_encoded', 'day_of_week_encoded'
]

X = df_features[feature_cols].fillna(df_features[feature_cols].mean())
y = df_features['quantity_sold']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Training samples : {len(X_train)}")
print(f"Testing  samples : {len(X_test)}")


In [ ]:
# Linear Regression
lr_model  = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)

lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
lr_r2   = r2_score(y_test, y_pred_lr)
lr_mae  = mean_absolute_error(y_test, y_pred_lr)
print(f"Linear Regression — RMSE: {lr_rmse:.2f}  R2: {lr_r2:.4f}  MAE: {lr_mae:.2f}")


In [ ]:
# Random Forest
rf_model  = RandomForestRegressor(n_estimators=100, max_depth=15,
                                   random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_r2   = r2_score(y_test, y_pred_rf)
rf_mae  = mean_absolute_error(y_test, y_pred_rf)
print(f"Random Forest     — RMSE: {rf_rmse:.2f}  R2: {rf_r2:.4f}  MAE: {rf_mae:.2f}")


In [ ]:
# LightGBM
lgb_model = lgb.LGBMRegressor(
    n_estimators=200, learning_rate=0.05, max_depth=7,
    num_leaves=31, random_state=42, verbose=-1, n_jobs=-1
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False)]
)
y_pred_lgb = lgb_model.predict(X_test)

lgb_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lgb))
lgb_r2   = r2_score(y_test, y_pred_lgb)
lgb_mae  = mean_absolute_error(y_test, y_pred_lgb)
print(f"LightGBM          — RMSE: {lgb_rmse:.2f}  R2: {lgb_r2:.4f}  MAE: {lgb_mae:.2f}")


In [ ]:
models_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'LightGBM'],
    'RMSE':  [lr_rmse,  rf_rmse,  lgb_rmse],
    'R2':    [lr_r2,    rf_r2,    lgb_r2],
    'MAE':   [lr_mae,   rf_mae,   lgb_mae]
})
print(models_df.round(4).to_string(index=False))

best_model       = lgb_model
best_predictions = y_pred_lgb
best_r2          = lgb_r2
print("\n✅ Selected: LightGBM")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for col, model_name, y_pred, color in [
    (0, 'Random Forest', y_pred_rf,  'blue'),
    (1, 'LightGBM',      y_pred_lgb, 'green'),
]:
    axes[0, col].scatter(y_test, y_pred, alpha=0.4, color=color, s=15)
    mn, mx = y_test.min(), y_test.max()
    axes[0, col].plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect')
    axes[0, col].set(title=f'{model_name}: Actual vs Predicted',
                     xlabel='Actual', ylabel='Predicted')
    axes[0, col].legend()

    residuals = y_test - y_pred
    axes[1, col].hist(residuals, bins=30, color=color, edgecolor='black', alpha=0.7)
    axes[1, col].axvline(0, color='red', ls='--', lw=2)
    axes[1, col].set(title=f'{model_name}: Residuals',
                     xlabel='Residual', ylabel='Frequency')

plt.suptitle('Model Evaluation', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Step 7 — Feature Importance <a id='step7'></a>

Which variables drive demand predictions the most?


In [ ]:
feature_importance = (
    pd.DataFrame({'feature':    feature_cols,
                  'importance': best_model.feature_importances_})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)
print(feature_importance.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(feature_importance['feature'], feature_importance['importance'],
             color='steelblue')
axes[0].invert_yaxis()
axes[0].set(title='Feature Importance (LightGBM)', xlabel='Importance Score')

axes[1].pie(feature_importance['importance'],
            labels=feature_importance['feature'],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Importance Distribution')

plt.suptitle('What Drives Customer Demand?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Step 8 — Price Optimization Algorithm <a id='step8'></a>

For each product we:
1. Sweep prices from 50 % to 200 % of the current price  
2. Predict demand at each price using LightGBM  
3. Compute `revenue = price × quantity` and `profit = (price − cost) × quantity`  
4. Pick the price that **maximises revenue**

> 💡 *Adjust `COST_PER_UNIT` below to match your actual unit cost.*


In [ ]:
COST_PER_UNIT = 50   # ← change this to your actual unit cost

def optimize_price_for_product(product_row, price_model,
                                price_range=(10, 300), n_steps=50):
    """
    Sweep a price range and find the revenue-maximising price.
    product_row : a single-row Series or dict containing all feature values
    """
    test_prices = np.linspace(price_range[0], price_range[1], n_steps)
    results = []

    for test_price in test_prices:
        row = product_row.copy()
        row['price']              = test_price
        row['price_ratio']        = test_price / max(row['competitor_price'], 0.01)
        row['price_gap']          = row['competitor_price'] - test_price
        row['price_premium']      = (test_price - row['base_price']) / max(row['base_price'], 0.01)
        row['marketing_per_unit'] = row['marketing_spend'] / 1   # placeholder unit

        X_row    = np.array([row[f] for f in feature_cols]).reshape(1, -1)
        pred_qty = max(price_model.predict(X_row)[0], 5)

        results.append({
            'price':    test_price,
            'quantity': pred_qty,
            'revenue':  test_price * pred_qty,
            'profit':   (test_price - COST_PER_UNIT) * pred_qty
        })

    results_df = pd.DataFrame(results)
    best_idx   = results_df['revenue'].idxmax()

    return {
        'optimal_price':      results_df.loc[best_idx, 'price'],
        'predicted_quantity': results_df.loc[best_idx, 'quantity'],
        'predicted_revenue':  results_df.loc[best_idx, 'revenue'],
        'predicted_profit':   results_df.loc[best_idx, 'profit'],
        'results':            results_df
    }

print("✅ Optimization function defined.")


In [ ]:
print("Generating recommendations for all products…")
recommendations = []

for product_id in df_features['product_id'].unique():
    row = df_features[df_features['product_id'] == product_id].iloc[-1].copy()

    cur_price    = row['price']
    cur_quantity = row['quantity_sold']
    cur_revenue  = row['revenue']
    cur_profit   = row['profit']

    opt = optimize_price_for_product(
        row, best_model,
        price_range=(cur_price * 0.5, cur_price * 2)
    )

    rev_lift_pct    = ((opt['predicted_revenue'] - cur_revenue) / cur_revenue * 100
                       if cur_revenue > 0 else 0)
    profit_lift_pct = ((opt['predicted_profit']  - cur_profit)  / abs(cur_profit) * 100
                       if cur_profit != 0 else 0)

    recommendations.append({
        'product_id':           product_id,
        'current_price':        round(cur_price, 2),
        'optimal_price':        round(opt['optimal_price'], 2),
        'price_change_pct':     round((opt['optimal_price'] - cur_price) / cur_price * 100, 2),
        'current_quantity':     int(cur_quantity),
        'projected_quantity':   int(opt['predicted_quantity']),
        'current_revenue':      round(cur_revenue, 2),
        'projected_revenue':    round(opt['predicted_revenue'], 2),
        'revenue_lift_pct':     round(rev_lift_pct, 2),
        'current_profit':       round(cur_profit, 2),
        'projected_profit':     round(opt['predicted_profit'], 2),
        'profit_lift_pct':      round(profit_lift_pct, 2),
    })

rec_df = pd.DataFrame(recommendations)
print(f"\nRecommendations generated for {len(rec_df)} products")
rec_df.head(10)


In [ ]:
rec_df.to_csv('price_optimization_recommendations.csv', index=False)
print("✅ Saved: price_optimization_recommendations.csv")


---
## Step 9 — Visualize Optimization Curves <a id='step9'></a>

Revenue and quantity curves across the price sweep for the first 9 products.  
⭐ = optimal price · green dashed = optimal · orange dashed = current price.


In [ ]:
sample_products = df_features['product_id'].unique()[:9]
fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, product_id in enumerate(sample_products):
    row       = df_features[df_features['product_id'] == product_id].iloc[-1].copy()
    cur_price = row['price']

    opt   = optimize_price_for_product(row, best_model,
                                       price_range=(cur_price * 0.5, cur_price * 2))
    sweep = opt['results']

    ax  = axes[idx]
    ax2 = ax.twinx()

    line1 = ax.plot(sweep['price'], sweep['revenue'], 'b-', lw=2, label='Revenue')
    ax.fill_between(sweep['price'], sweep['revenue'], alpha=0.15, color='blue')
    line2 = ax2.plot(sweep['price'], sweep['quantity'], 'r--', lw=2, label='Quantity')

    best_idx  = sweep['revenue'].idxmax()
    opt_price = sweep.loc[best_idx, 'price']
    opt_rev   = sweep.loc[best_idx, 'revenue']

    ax.scatter([opt_price], [opt_rev], color='blue', s=120, zorder=5, marker='*')
    ax.axvline(opt_price, color='green',  ls='--', alpha=0.7, label='Optimal')
    ax.axvline(cur_price, color='orange', ls='--', alpha=0.7, label='Current')

    ax.set(title=f'Product {int(product_id)}: Optimization Curve',
           xlabel='Price ($)', ylabel='Revenue ($)')
    ax2.set_ylabel('Quantity Sold', color='red')
    ax.tick_params(axis='y', labelcolor='blue')
    ax2.tick_params(axis='y', labelcolor='red')

    lines  = line1 + line2
    labels = [l.get_label() for l in lines]
    ax.legend(lines, labels, loc='upper left', fontsize=7)

plt.suptitle('Revenue & Quantity vs Price — Sample Products',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Step 10 — Business Impact Analysis <a id='step10'></a>

Portfolio-level view: total revenue and profit improvement across all products.


In [ ]:
total_cur_rev   = rec_df['current_revenue'].sum()
total_proj_rev  = rec_df['projected_revenue'].sum()
rev_lift        = total_proj_rev - total_cur_rev
rev_lift_pct    = rev_lift / total_cur_rev * 100

total_cur_prof  = rec_df['current_profit'].sum()
total_proj_prof = rec_df['projected_profit'].sum()
prof_lift       = total_proj_prof - total_cur_prof
prof_lift_pct   = prof_lift / abs(total_cur_prof) * 100

price_ups = rec_df[rec_df['price_change_pct'] > 0]
price_dns = rec_df[rec_df['price_change_pct'] < 0]

print(f"Products analysed        : {len(rec_df)}")
print(f"Current  Revenue (total) : ${total_cur_rev:>12,.2f}")
print(f"Projected Revenue (total): ${total_proj_rev:>12,.2f}  (+{rev_lift_pct:.2f}%)")
print(f"\nCurrent  Profit  (total) : ${total_cur_prof:>12,.2f}")
print(f"Projected Profit  (total): ${total_proj_prof:>12,.2f}  (+{prof_lift_pct:.2f}%)")
print(f"\nProducts to INCREASE price: {len(price_ups)}"
      f"  (avg +{price_ups['price_change_pct'].mean():.1f}%)")
print(f"Products to DECREASE price: {len(price_dns)}"
      f"  (avg {price_dns['price_change_pct'].mean():.1f}%)")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Sort by product_id for clean x-axis
rec_plot = rec_df.sort_values('product_id').reset_index(drop=True)
x     = np.arange(len(rec_plot))
width = 0.35

axes[0, 0].bar(x - width/2, rec_plot['current_revenue'],   width,
               label='Current',   color='coral',      alpha=0.8)
axes[0, 0].bar(x + width/2, rec_plot['projected_revenue'], width,
               label='Projected', color='lightgreen', alpha=0.8)
axes[0, 0].set(title='Revenue: Current vs Projected',
               xlabel='Product ID', ylabel='Revenue ($)')
axes[0, 0].set_xticks(x[::max(1, len(x)//20)])
axes[0, 0].set_xticklabels(rec_plot['product_id'].iloc[::max(1, len(x)//20)].astype(int),
                            rotation=45)
axes[0, 0].legend()

colors_lift = ['green' if v > 0 else 'red' for v in rec_plot['revenue_lift_pct']]
axes[0, 1].barh(rec_plot['product_id'].astype(str),
                rec_plot['revenue_lift_pct'], color=colors_lift, alpha=0.7)
axes[0, 1].axvline(0, color='black', lw=0.8)
axes[0, 1].set(title='Revenue Lift % by Product', xlabel='Revenue Lift (%)')
axes[0, 1].set_yticks(axes[0, 1].get_yticks()[::max(1, len(rec_plot)//20)])

colors_price = ['blue' if v > 0 else 'orange' for v in rec_plot['price_change_pct']]
axes[1, 0].barh(rec_plot['product_id'].astype(str),
                rec_plot['price_change_pct'], color=colors_price, alpha=0.7)
axes[1, 0].axvline(0, color='black', lw=0.8)
axes[1, 0].set(title='Recommended Price Changes (%)', xlabel='Price Change (%)')
axes[1, 0].set_yticks(axes[1, 0].get_yticks()[::max(1, len(rec_plot)//20)])

axes[1, 1].scatter(rec_plot['current_profit'], rec_plot['projected_profit'],
                   s=100, alpha=0.7, color='purple')
mn = min(rec_plot['current_profit'].min(), rec_plot['projected_profit'].min())
mx = max(rec_plot['current_profit'].max(), rec_plot['projected_profit'].max())
axes[1, 1].plot([mn, mx], [mn, mx], 'r--', lw=2, label='No change')
axes[1, 1].set(title='Current vs Projected Profit',
               xlabel='Current Profit ($)', ylabel='Projected Profit ($)')
axes[1, 1].legend()

plt.suptitle('Business Impact of Price Optimization',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Step 11 — Key Insights & Summary <a id='step11'></a>


In [ ]:
top3 = feature_importance['feature'].head(3).tolist()
sep = '=' * 70
print(sep)
print('   KEY INSIGHTS & RECOMMENDATIONS')
print(sep)
print(f'\n1. DATASET')
print(f'   Rows     : {len(df):,}')
print(f'   Products : {df["product_id"].nunique()}')
print(f'   Weeks    : {df["week"].min()} - {df["week"].max()}')
print(f'\n2. BEST DEMAND MODEL')
print(f'   R-squared : {best_r2:.4f}')
print(f'   Top drivers: {", ".join(top3)}')
print(f'\n3. PRICE ELASTICITY')
print(f'   Avg elasticity : {df_features["price_elasticity"].mean():.2f}')
print('   Low-elasticity = candidates for price increase.')
print(f'\n4. REVENUE OPTIMIZATION')
print(f'   Revenue uplift : {rev_lift_pct:.2f}%  (${rev_lift:,.0f} additional)')
print(f'   Profit  uplift : {prof_lift_pct:.2f}%  (${prof_lift:,.0f} additional)')
print(f'\n5. PRICING STRATEGY SPLIT')
print(f'   Increase price : {len(price_ups)} products')
print(f'   Decrease price : {len(price_dns)} products')
print(f'\n6. ROADMAP')
print('   1. Start with low-elasticity products')
print('   2. A/B test before full rollout')
print('   3. Monitor demand weekly')
print('   4. Retrain model quarterly')
print('\n   -> Saved: price_optimization_recommendations.csv')
print('\n' + sep)
print('   NOTEBOOK COMPLETE')
print(sep)
